## Will use MobileNet's frozen layers for feature extraction and then classify..

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
import tensorflow as tf
from tensorflow.keras import layers, Sequential
from tensorflow.keras.utils import image_dataset_from_directory

In [3]:
train_ds = image_dataset_from_directory(directory = 'dataset/train', label_mode='int', image_size=(224,224) ,shuffle=True,color_mode="rgb")

Found 2484 files belonging to 432 classes.


In [4]:
test_ds = image_dataset_from_directory(directory = 'dataset/val', label_mode='int', image_size=(224,224) ,shuffle=False,color_mode="rgb")

Found 860 files belonging to 432 classes.


In [5]:
len(train_ds.class_names)

432

In [6]:
for x,y in train_ds:
    print(f'Exact size of input images : {x.shape}') # (batch_size,x,y,channels)
    break

Exact size of input images : (32, 224, 224, 3)


In [7]:
from tensorflow.keras.applications.mobilenet_v2 import MobileNetV2, preprocess_input

In [8]:
base_model = MobileNetV2(    input_shape=(224,224,3), # this is the input tensor here, so no layers.Input() in seq. model
    alpha=1.0,
    include_top=False,
    weights='imagenet',
    input_tensor=None,
    pooling=None,
    classes=432,
    classifier_activation='softmax',
    name=None,
)

## data augmentation...

In [9]:
data_aug = Sequential([
        layers.RandomRotation(0.02),      # very small tilt
    layers.RandomZoom(0.05),           # small zoom
    layers.RandomTranslation(0.05, 0.05),  # slight misalignment
])

In [10]:
from tensorflow.keras.callbacks import EarlyStopping

early_stop = EarlyStopping(
    monitor='val_loss',  
    patience=5,          
    restore_best_weights=True  # restores model weights with the best value 
)

In [11]:
model = Sequential([
    
    layers.Input(shape=(224,224,3)),
    data_aug,
    
    layers.Lambda(preprocess_input),
    base_model,

    layers.GlobalAveragePooling2D(),
    layers.BatchNormalization(),

    layers.Dense(512, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.4),
    
    layers.Dense(432, activation='softmax')
])

In [12]:
model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ sequential (Sequential)              │ (None, 224, 224, 3)         │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ lambda (Lambda)                      │ (None, 224, 224, 3)         │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ mobilenetv2_1.00_224 (Functional)    │ (None, 7, 7, 1280)          │       2,257,984 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ global_average_pooling2d             │ (None, 1280)                │               0 │
│ (GlobalAveragePooling2D)             │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization                  │ (None, 1280)                │           5,120 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ (None, 512)                 │         655,872 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_1                │ (None, 512)                 │           2,048 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout (Dropout)                    │ (None, 512)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ (None, 432)                 │         221,616 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 3,142,640 (11.99 MB)

 Trainable params: 3,104,944 (11.84 MB)

 Non-trainable params: 37,696 (147.25 KB)

In [13]:
base_model.trainable = False

In [14]:
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy' ,metrics = ['accuracy'])

In [15]:
model.fit(train_ds, batch_size=32, epochs=10,callbacks=[early_stop], validation_data=test_ds)

Epoch 1/10
78/78 ━━━━━━━━━━━━━━━━━━━━ 48s 561ms/step - accuracy: 0.0105 - loss: 6.2302 - val_accuracy: 0.0337 - val_loss: 5.5631
Epoch 2/10
78/78 ━━━━━━━━━━━━━━━━━━━━ 95s 737ms/step - accuracy: 0.1304 - loss: 4.4135 - val_accuracy: 0.0651 - val_loss: 4.9889
Epoch 3/10
78/78 ━━━━━━━━━━━━━━━━━━━━ 68s 555ms/step - accuracy: 0.2725 - loss: 3.3433 - val_accuracy: 0.0756 - val_loss: 4.5317
Epoch 4/10
78/78 ━━━━━━━━━━━━━━━━━━━━ 85s 596ms/step - accuracy: 0.4231 - loss: 2.5486 - val_accuracy: 0.1035 - val_loss: 4.2527
Epoch 5/10
78/78 ━━━━━━━━━━━━━━━━━━━━ 98s 800ms/step - accuracy: 0.5491 - loss: 1.9692 - val_accuracy: 0.1360 - val_loss: 3.9996
Epoch 6/10
78/78 ━━━━━━━━━━━━━━━━━━━━ 83s 816ms/step - accuracy: 0.6663 - loss: 1.5039 - val_accuracy: 0.1442 - val_loss: 3.9566
Epoch 7/10
78/78 ━━━━━━━━━━━━━━━━━━━━ 46s 597ms/step - accuracy: 0.7432 - loss: 1.1628 - val_accuracy: 0.1593 - val_loss: 3.9107
Epoch 8/10
78/78 ━━━━━━━━━━━━━━━━━━━━ 51s 660ms/step - accuracy: 0.8104 - loss: 0.9104 - val_accu

In [16]:
base_model.trainable = True

for layer in base_model.layers[:-10]:
    layer.trainable = False

for layer in base_model.layers[-10:]:
    layer.trainable = True

for layer in base_model.layers:
    if isinstance(layer, tf.keras.layers.BatchNormalization):
        layer.trainable = False    

In [17]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-5),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.fit(train_ds, epochs=10, callbacks=[early_stop], validation_data=test_ds)

Epoch 1/10
78/78 ━━━━━━━━━━━━━━━━━━━━ 58s 679ms/step - accuracy: 0.8482 - loss: 0.7851 - val_accuracy: 0.1477 - val_loss: 3.9292
Epoch 2/10
78/78 ━━━━━━━━━━━━━━━━━━━━ 50s 642ms/step - accuracy: 0.8619 - loss: 0.7344 - val_accuracy: 0.1442 - val_loss: 3.9127
Epoch 3/10
78/78 ━━━━━━━━━━━━━━━━━━━━ 51s 654ms/step - accuracy: 0.8704 - loss: 0.7121 - val_accuracy: 0.1372 - val_loss: 3.9314
Epoch 4/10
78/78 ━━━━━━━━━━━━━━━━━━━━ 76s 573ms/step - accuracy: 0.8728 - loss: 0.7116 - val_accuracy: 0.1407 - val_loss: 3.9077
Epoch 5/10
78/78 ━━━━━━━━━━━━━━━━━━━━ 83s 591ms/step - accuracy: 0.8909 - loss: 0.6597 - val_accuracy: 0.1442 - val_loss: 3.8549
Epoch 6/10
78/78 ━━━━━━━━━━━━━━━━━━━━ 86s 644ms/step - accuracy: 0.8857 - loss: 0.6645 - val_accuracy: 0.1477 - val_loss: 3.8365
Epoch 7/10
78/78 ━━━━━━━━━━━━━━━━━━━━ 53s 686ms/step - accuracy: 0.8957 - loss: 0.6378 - val_accuracy: 0.1465 - val_loss: 3.8453
Epoch 8/10
78/78 ━━━━━━━━━━━━━━━━━━━━ 54s 696ms/step - accuracy: 0.8990 - loss: 0.6428 - val_accu

In [18]:
model.evaluate(test_ds)

27/27 ━━━━━━━━━━━━━━━━━━━━ 9s 330ms/step - accuracy: 0.1453 - loss: 3.8297 


[3.829663038253784, 0.14534883201122284]

In [19]:
images, labels = next(iter(train_ds))
print(labels.shape)
print(labels[:5])

(32,)
tf.Tensor([352 289 402 124 355], shape=(5,), dtype=int32)


In [20]:
for images, labels in test_ds.take(1):
    preds = model.predict(images)
    print("True:", labels[:20].numpy())
    print("Pred:", tf.argmax(preds, axis=1)[:10].numpy())

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step
True: [0 0 0 0 1 1 2 2 3 3 4 4 5 5 6 6 7 7 8 8]
Pred: [ 72 396  72 396  25  37 354 216   2 316]


In [21]:
print(test_ds.class_names)

['000', '001', '002', '003', '004', '005', '006', '007', '008', '009', '010', '011', '012', '013', '014', '015', '016', '017', '018', '019', '020', '021', '022', '023', '024', '025', '026', '027', '028', '029', '030', '031', '032', '033', '034', '035', '036', '037', '038', '039', '040', '041', '042', '043', '044', '045', '046', '047', '048', '049', '050', '051', '052', '053', '054', '055', '056', '057', '058', '059', '060', '061', '062', '063', '064', '065', '066', '067', '068', '069', '070', '071', '072', '073', '074', '075', '076', '077', '078', '079', '080', '081', '082', '083', '084', '085', '086', '087', '088', '089', '090', '091', '092', '093', '094', '095', '096', '097', '098', '099', '100', '101', '102', '103', '104', '105', '106', '107', '108', '109', '110', '111', '112', '113', '114', '115', '116', '117', '118', '119', '120', '121', '122', '123', '124', '125', '126', '127', '128', '129', '130', '131', '132', '133', '134', '135', '136', '137', '138', '139', '140', '141', '142'

In [22]:
import numpy as np

images, labels = next(iter(test_ds.shuffle(1000)))
preds = model.predict(images)

# check confidence
for i in range(5):
    pred_probs = preds[i]
    top5 = np.argsort(pred_probs)[-5:][::-1]

    print(f"\nTrue: {labels[i].numpy()}")
    print("Top-5 predictions:", top5)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 320ms/step

True: 15
Top-5 predictions: [248  54 242 206 303]

True: 15
Top-5 predictions: [120   3   2 305  14]

True: 16
Top-5 predictions: [250 125  16 192  17]

True: 16
Top-5 predictions: [196 124 197 244 136]

True: 17
Top-5 predictions: [113 305 125  89  53]
